In [133]:
import os
import numpy as np
import xarray as xr
import pandas as pd
import json

In [2]:
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [3]:
from lib import adwin
from lib.dataimport import (
    import_adwin_rcvd_traces,
    import_adwin_sent_traces,
    import_ordered_traces,
    import_results_pickle,
)
from lib.trace import OrderedTrace, TraceComparison

In [128]:
from dataclasses import dataclass, field
from typing import Any


@dataclass
class Span:
    trace_id: str
    span_id: str
    operation: str

    start_time: float
    end_time: float

    tags: dict[str, Any] = field(default_factory=dict)
    events: list[tuple[float, str]] = field(default_factory=list)

    @property
    def duration(self) -> float:
        return self.end_time - self.start_time

@dataclass
class Trace:
    trace_id: str
    spans: dict[str, Span] = field(default_factory=dict)
    graph: dict[str, list[str]] = field(default_factory=dict)
    processes: dict[str, list[str]] = field(default_factory=dict)

In [10]:
HOST_EVENT_TYPES = [
    "SUBROUTINE_SEND_ATTEMPT",
    "SUBROUTINE_SENT",
    "RESULT_RCVD",
    "CLAS_MSG_SENT",
    "CLAS_MSG_RCVD",
]

QNODEOS_EVENT_TYPES = [
    "SCHEDULER_ARRIVE_USER_PROCESS",
    "SCHEDULER_SCHEDULE_USER_PROCESS",
    "SCHEDULER_ARRIVE_NET_PROCESS",
    "SCHEDULER_SCHEDULE_NET_PROCESS",
    "SCHEDULER_HIGH_PRIO_PROCESS_WAITS_ON_LOWER_PRIO",
    "PROCMGR_SUBROUTINE_ADDED_P0",
    "PROCMGR_SUBROUTINE_DONE_P0",
    "PROCESSOR_START_USER_PROCESS",
    "PROCESSOR_WAIT_USER_PROCESS",
    "PROCESSOR_FINISH_USER_PROCESS",
    "PROCESSOR_START_NET_PROCESS",
    "PROCESSOR_FINISH_NET_PROCESS",
    "PROCESSOR_CONSUME_OUTCOME",
    "QDEVICE_PRODUCE_SQG_CMD",
    "QDEVICE_PRODUCE_ENT_CMD",
    "QDEVICE_PRODUCE_MSR_CMD",
    "QDEVICE_PRODUCE_OUTCOME",
    "QDEVICE_CONSUME_CMD",
    "QNETWORK_ENT_PULL",
    "EGP_NEI_OK",
    "EGP_ENT_OK",
]

In [109]:
# pip install netcdf4 h5netcdf

SPI_RESOLUTION = 10 # Duration of a single QNodeOS-QDevice communication cycle (us)
def load_data(traces_dir):
    # Host trace
    host_traces_ms = import_ordered_traces(traces_dir, filename_end=f"host_trace.csv", resolution_us=SPI_RESOLUTION)
    # QNodeOS traces
    qnodeos_traces_ms = import_ordered_traces(traces_dir, filename_end=f"qnodeos_trace.csv", resolution_us=SPI_RESOLUTION)
    # QDevice trace
    adwin_sent_traces = import_adwin_sent_traces(traces_dir, f"link_layer_sent.hdf5", SPI_RESOLUTION)
    adwin_recv_traces = import_adwin_rcvd_traces(traces_dir, f"link_layer_rcvd.hdf5", SPI_RESOLUTION)
    return host_traces_ms, qnodeos_traces_ms, adwin_sent_traces, adwin_recv_traces

In [52]:
# Client host trace:
#
# (1) SUBROUTINE_SENT
# (6) RESULT_RCVD
# (7) CLAS_MSG_SENT
def build_client_host_spans(client_trace, span_id=0):
    spans = []
    current = None
    for entry in client_trace.entries:
        if entry.event == "SUBROUTINE_SEND_ATTEMPT":
            current = Span(
                trace_id="",
                span_id=str(span_id),
                operation="Client_Host",
                start_time=entry.time,
                end_time=entry.time,
            )

        elif current is not None:
            current.events.append((entry.time, entry.event))

            if entry.event == "CLAS_MSG_SENT":
                current.end_time = entry.time
                spans.append(current)
                current = None
                span_id += 1
    return spans

In [168]:

# Client QNodeOS repeated events:
#
# (2) PROCMGR_SUBROUTINE_ADDED_P0
# (2a) QDEVICE_PRODUCE_ENT_CMD (repeated many times)
# (3) EGP_NEI_OK
# (3a) QDEVICE_PRODUCE_SQG_CMD (couple of times, depending on measurement basis)
# (4) QDEVICE_PRODUCE_MSR_CMD
# (5) PROCMGR_SUBROUTINE_DONE_P0

# Server QNodeOS repeated events:
#
# (2) PROCMGR_SUBROUTINE_ADDED_P0
# (2a) QDEVICE_PRODUCE_ENT_CMD (repeated many times)
# (3) EGP_NEI_OK
# (4) QDEVICE_PRODUCE_SQG_CMD (1 or 2 times, depending on Psi+ or Psi-)
# (5) PROCMGR_SUBROUTINE_DONE_P0
# (9) PROCMGR_SUBROUTINE_ADDED_P0
# (9a) QDEVICE_PRODUCE_SQG_CMD (couple of times, depending on measurement basis)
# (10) QDEVICE_PRODUCE_MSR_CMD
# (11) PROCMGR_SUBROUTINE_DONE_P0
def build_qnodeos_spans(qnodeos_trace, span_id=0, op=""):
    spans = []
    current = None
    for entry in qnodeos_trace.entries:
        if entry.event == "PROCMGR_SUBROUTINE_ADDED_P0":
            current = Span(trace_id="", span_id=str(span_id), operation=op, start_time=entry.time, end_time=entry.time)

        elif current is not None:
            #current.events.append((entry.time, entry.event))
            if entry.event == "EGP_NEI_OK":
                current.events.append((entry.time, entry.event))

            if entry.event == "PROCMGR_SUBROUTINE_DONE_P0":
                current.end_time = entry.time
                spans.append(current)
                current = None
                span_id += 1
    return spans

# Extra helper functions for checking if things make sense
def count_entangle_instructions(trace):
    cnt = 0
    for entry in trace.entries:
        if entry.event == "QDEVICE_PRODUCE_ENT_CMD":
            cnt += 1
    return cnt

def count_neiok_instructions(trace):
    cnt = 0
    for entry in trace.entries:
        if entry.event == "EGP_NEI_OK":
            cnt += 1
    return cnt

In [147]:
# Server host repeated events:
#
# (1) SUBROUTINE_SENT
# (6) RESULT_RCVD
# (7) CLAS_MSG_RCVD
# (8) SUBROUTINE_SENT
# (12) RESULT_RCVD
def build_server_host_spans(server_trace, span_id=0):
    spans = []
    span_events = []
    span_start = None
    result_rcvd_count = 0
    for e in server_trace.entries:
        t = float(e.time)
        ev = e.event

        # start span
        if span_start is None:
            span_start = t

        span_events.append((t, ev))

        # track progress
        if ev == "RESULT_RCVD":
            result_rcvd_count += 1

        # span boundary condition (YOUR STRUCTURE)
        if result_rcvd_count == 2:
            spans.append(
                Span(
                    trace_id="",
                    span_id=str(span_id),
                    operation="Server_Host",
                    start_time=span_start,
                    end_time=t,
                    events=span_events.copy(),
                )
            )

            # reset for next span
            span_id += 1
            span_events.clear()
            span_start = None
            result_rcvd_count = 0

    return spans

In [149]:
psi_plus_val = adwin.name_to_xarray_number_to_qnodeos("SUCCESS_PSI_PLUS")
psi_minus_val = adwin.name_to_xarray_number_to_qnodeos("SUCCESS_PSI_MINUS")
ent_fail_val = adwin.name_to_xarray_number_to_qnodeos("ENTANGLEMENT_FAILURE")
ent_sync_fail_val = adwin.name_to_xarray_number_to_qnodeos("ENTANGLEMENT_SYNC_FAILURE")
cmd_ent = adwin.name_to_xarray_number_from_qnodeos("ENTANGLE")
init_qubit = adwin.name_to_xarray_number_from_qnodeos("INIT_QUBIT")
cmd_measure = adwin.name_to_xarray_number_from_qnodeos("MEASURE")
# For a given ADwin trace, find all time windows in which it does entanglement attempts.
# Return a list of these windows, where each window is specified by the start and end
# index of the events in the trace.
def get_ent_gen_slices(trace: xr.DataArray):
    ent_gen_indices = np.where(np.isin(trace, [psi_plus_val, psi_minus_val]))[0]

    slice_indices = []  # list of [start, end) tuples
    start_idx = 0
    for idx in ent_gen_indices:
        if start_idx != idx:
            slic = trace.isel(time=slice(start_idx, idx + 1))
            attempt_indices = np.where(
                np.isin(slic, [ent_fail_val, ent_sync_fail_val])
            )[0]
            if len(attempt_indices) == 0:
                first_attempt_index = idx
            else:
                first_attempt_index = start_idx + attempt_indices[0]
            slice_indices.append((first_attempt_index, idx + 1))
        start_idx = idx + 1

    return slice_indices

def generate_device_spans(sent_trace, rcvd_trace, span_id=0):
    # Generate entanglement spans
    ent_gen_slices = get_ent_gen_slices(sent_trace)
    entanglement_spans = []
    
    for start, end in ent_gen_slices:
        sent_slic = sent_trace.isel(time=slice(start, end))
        rcvd_slic = rcvd_trace.isel(time=slice(start, end))
        assert sent_slic.shape == rcvd_slic.shape
        assert np.all(rcvd_slic == cmd_ent)
        if len(rcvd_slic.coords["time"].values) == 0:
            print(sent_slic.coords["time"].values)
        start_time = rcvd_slic.coords["time"].values[0]
        end_time = sent_slic.coords["time"].values[-1]
        span = Span(trace_id="", span_id=str(span_id), operation="Entanglement Generation", start_time=start_time, end_time=end_time)
        entanglement_spans.append(span)
        span_id += 1

    return entanglement_spans

In [182]:
def trace_to_jaeger_json(trace: Trace, service_name="qnodeos"):
    parents = {}
    processes = {}

    for parent, children in trace.graph.items():
        for child in children:
            parents[child] = parent

    for process, children in trace.processes.items():
        for child in children:
            processes[child] = process

    jaeger_spans = []

    for span in trace.spans.values():
        refs = []

        if span.span_id in parents:
            refs.append(
                {
                    "refType": "CHILD_OF",
                    "traceID": trace.trace_id,
                    "spanID": parents[span.span_id],
                }
            )

        tags = [
            {
                "key": str(k),
                "type": "string",
                "value": str(v),
            }
            for k, v in span.tags.items()
        ]

        logs = [
            {
                "timestamp": int(t),
                "fields": [
                    {
                        "key": "event",
                        "type": "string",
                        "value": event,
                    }
                ],
            }
            for t, event in span.events
        ]

        jaeger_spans.append(
            {
                "traceID": trace.trace_id,
                "spanID": span.span_id,
                "operationName": span.operation,
                "references": refs,
                "startTime": int(span.start_time * 1e3),
                "duration": int(span.duration*1e3),
                "tags": tags,
                "logs": logs,
                "processID": processes[span.span_id],
                "warnings": None,
            }
        )

    all_processes = {}
    for p in trace.processes:
        all_processes[p] = {"serviceName": p, "tags": []}

    return {
        "data": [
            {
                "traceID": trace.trace_id,
                "spans": jaeger_spans,
                "processes": all_processes,
                "warnings": None,
            }
        ],
        "total": 1,
        "limit": 0,
        "offset": 0,
        "errors": None,
    }

def export_trace(trace: Trace, filename: str):
    with open(filename, "w") as f:
        json.dump(
            trace_to_jaeger_json(trace),
            f,
            indent=2,
        )

In [111]:
EXP_NAME = "dqc_20240304"
PARAM_COMBO = "a2t2"
NUM_REPS = 3
DATA_DIR = "alpha2_theta2"
DATA_PATH = "../data"

server_dir = os.path.join(DATA_PATH, EXP_NAME, DATA_DIR, "LT3")
client_dir = os.path.join(DATA_PATH, EXP_NAME, DATA_DIR, "LT4")
server_results = import_results_pickle(server_dir, allow_multiple=True)
client_results = import_results_pickle(client_dir, allow_multiple=True)

client_host_traces, client_qnodeos_traces, client_adwin_sent_traces, client_adwin_recv_traces = load_data(client_dir)
server_host_traces, server_qnodeos_traces, server_adwin_sent_traces, server_adwin_recv_traces = load_data(server_dir)

looking in dir ../data/dqc_20240304/alpha2_theta2/LT3 for files with name *results.pickle
looking in dir ../data/dqc_20240304/alpha2_theta2/LT4 for files with name *results.pickle
looking in dir ../data/dqc_20240304/alpha2_theta2/LT4 for files with name *host_trace.csv
files found: [PosixPath('../data/dqc_20240304/alpha2_theta2/LT4/20240304_211807_incomplete_bqc_client_client_host_trace.csv'), PosixPath('../data/dqc_20240304/alpha2_theta2/LT4/20240304_212551_incomplete_bqc_client_client_host_trace.csv'), PosixPath('../data/dqc_20240304/alpha2_theta2/LT4/20240304_214242_incomplete_bqc_client_client_host_trace.csv')]
looking in dir ../data/dqc_20240304/alpha2_theta2/LT4 for files with name *qnodeos_trace.csv
files found: [PosixPath('../data/dqc_20240304/alpha2_theta2/LT4/20240304_211807_incomplete_bqc_client_client_qnodeos_trace.csv'), PosixPath('../data/dqc_20240304/alpha2_theta2/LT4/20240304_212551_incomplete_bqc_client_client_qnodeos_trace.csv'), PosixPath('../data/dqc_20240304/alpha2

In [191]:
HOST_QNODE_LAT=1e-3 # ms = 1us
QNODE_QDEV_LAT=1e-2 # ms = 10us
for rep in range(NUM_REPS):
    span_id = 0
    client_host_spans = build_client_host_spans(client_host_traces_ms[rep], span_id)
    span_id += len(client_host_spans)
    client_qnodeos_spans = build_qnodeos_spans(client_qnodeos_traces_ms[rep], span_id=span_id,op="Client_QNodeOS")
    span_id += len(client_qnodeos_spans)
    
    server_host_spans = build_server_host_spans(server_host_traces_ms[rep], span_id=span_id)
    span_id += len(server_host_spans)
    server_qnodeos_spans = build_qnodeos_spans(server_qnodeos_traces_ms[rep], op="Server_QNodeOS", span_id=span_id)
    span_id += len(server_qnodeos_spans)
    client_entanglement_spans = generate_device_spans(client_adwin_sent_traces[rep], client_adwin_recv_traces[rep], span_id=span_id)
    span_id += len(client_entanglement_spans)
    server_entanglement_spans = generate_device_spans(server_adwin_sent_traces[rep], server_adwin_recv_traces[rep], span_id=span_id)
    print(count_neiok_instructions(server_qnodeos_traces_ms[rep]))

    # Each experiment has 1 server_host_span, 1 client_host_span, 1 client_qnodeos_span, 2 server_qnodeos_spans
    # Each qnodeos_span contains 1 qdevice span
    # For every entanglement instruction request there is a corresponding event in adwin_receive and a result in adwin_sent
    # We only care about the 1st entanglement instruction sent to the device and the first nei ok response we get.
    # Let's get all the entanglement spans from the device traces
    traces = []
    j = 0
    for i in range(len(server_host_spans)):
        t = Trace(trace_id=f"trace{i+1}")
        server_host_span = server_host_spans[i]
        client_host_span = client_host_spans[i]
        server_qnodeos_span1 = server_qnodeos_spans[j]
        server_qnodeos_span2 = server_qnodeos_spans[j+1]
        client_qnodeos_span = client_qnodeos_spans[i]
        server_entanglement_span = server_entanglement_spans[i]
        client_entanglement_span = client_entanglement_spans[i]

        # After we are done with time-syncing things, this should be the difference between the start times
        offset_between_qdevices = server_entanglement_span.start_time - client_entanglement_span.start_time
        # Calculate offsets from qnodeos to qdevice
        if len(server_qnodeos_span1.events) != 1:
            print("qnodeos span missing entanglement ok log entry: ", len(server_qnodeos_span1.events))
            j += 2
            continue
        server_ent_offset = server_qnodeos_span1.end_time - server_qnodeos_span1.events[0][0]
        client_ent_offset =  client_qnodeos_span.end_time - client_qnodeos_span.events[0][0]
        # Align host and qnodeos traces - Assume a host<->qnodeos latency
        server_qnodeos1_duration = server_qnodeos_span1.duration
        server_qnodeos2_duration = server_qnodeos_span2.duration
        qnodeos_spans_offset = server_qnodeos_span2.start_time - server_qnodeos_span1.start_time
        server_qnodeos_span1_offset = server_host_span.start_time + HOST_QNODE_LAT - server_qnodeos_span1.start_time
        server_qnodeos_span1.start_time = server_host_span.start_time + HOST_QNODE_LAT
        server_qnodeos_span1.end_time = server_qnodeos1_duration + server_qnodeos_span1.start_time
        server_qnodeos_span2_offset = server_qnodeos1_duration + server_qnodeos_span1.start_time - server_qnodeos_span2.start_time
        server_qnodeos_span2.start_time = server_qnodeos_span1.start_time + qnodeos_spans_offset
        server_qnodeos_span2.end_time = server_qnodeos_span2.start_time + server_qnodeos2_duration

        client_qnodeos_duration = client_qnodeos_span.duration
        client_qnodeos_offset = client_host_span.start_time + HOST_QNODE_LAT - client_qnodeos_span.start_time
        client_qnodeos_span.start_time = client_host_span.start_time + HOST_QNODE_LAT
        client_qnodeos_span.end_time = client_qnodeos_span.start_time + client_qnodeos_duration

        # Align qnodeos<->device traces
        server_entanglement_dur = server_entanglement_span.duration
        client_entanglement_dur = client_entanglement_span.duration

        client_entanglement_span.end_time = client_qnodeos_span.end_time - QNODE_QDEV_LAT - client_ent_offset
        client_entanglement_span.start_time = client_qnodeos_span.end_time - client_entanglement_dur - client_ent_offset

        server_entanglement_span.end_time = server_qnodeos_span1.end_time - QNODE_QDEV_LAT  - server_ent_offset
        server_entanglement_span.start_time = server_qnodeos_span1.end_time - server_entanglement_dur - server_ent_offset

        # Align both client and server now
        observed_delta = server_entanglement_span.start_time - client_entanglement_span.start_time
        offset = offset_between_qdevices - observed_delta

        client_entanglement_span.start_time = client_entanglement_span.start_time - offset
        client_entanglement_span.end_time = client_entanglement_span.end_time - offset
        client_qnodeos_span.start_time = client_qnodeos_span.start_time - offset
        client_qnodeos_span.end_time = client_qnodeos_span.end_time - offset
        client_host_span.start_time = client_host_span.start_time - offset
        client_host_span.end_time = client_host_span.end_time - offset

        # TODO: Fix event offsets
        
        t.spans[server_host_span.span_id] = server_host_span
        t.spans[client_host_span.span_id] = client_host_span
        t.spans[server_qnodeos_span1.span_id] = server_qnodeos_span1
        t.spans[server_qnodeos_span2.span_id] = server_qnodeos_span2
        t.spans[client_qnodeos_span.span_id] = client_qnodeos_span
        t.spans[server_entanglement_span.span_id] = server_entanglement_span
        t.spans[client_entanglement_span.span_id] = client_entanglement_span
        t.graph[server_qnodeos_span1.span_id] = [server_entanglement_span.span_id]
        t.graph[client_qnodeos_span.span_id] = [client_entanglement_span.span_id]
        t.graph[server_host_span.span_id] = [server_qnodeos_span1.span_id, server_qnodeos_span2.span_id]
        t.graph[client_host_span.span_id] = [client_qnodeos_span.span_id]
        t.processes["server_host"] = [server_host_span.span_id]
        t.processes["client_host"] = [client_host_span.span_id]
        t.processes["server_qdevice"] = [server_entanglement_span.span_id]
        t.processes["client_qdevice"] = [client_entanglement_span.span_id]
        t.processes["server_qnodeos"] = [server_qnodeos_span1.span_id, server_qnodeos_span2.span_id]
        t.processes["client_qnodeos"] = [client_qnodeos_span.span_id]
        j += 2
        traces += [t]

    print(len(traces))
    shot = 0
    for t in traces:
        os.makedirs("traces", exist_ok=True)
        filename = os.path.join("traces", PARAM_COMBO + f"_{rep}_{shot}.json")
        export_trace(t, filename)
        shot += 1

400
qnodeos span missing entanglement ok log entry:  0
qnodeos span missing entanglement ok log entry:  0
qnodeos span missing entanglement ok log entry:  0
qnodeos span missing entanglement ok log entry:  0
qnodeos span missing entanglement ok log entry:  0
qnodeos span missing entanglement ok log entry:  0
qnodeos span missing entanglement ok log entry:  0
qnodeos span missing entanglement ok log entry:  0
qnodeos span missing entanglement ok log entry:  0
391
400
qnodeos span missing entanglement ok log entry:  0
qnodeos span missing entanglement ok log entry:  0
qnodeos span missing entanglement ok log entry:  0
qnodeos span missing entanglement ok log entry:  0
qnodeos span missing entanglement ok log entry:  0
qnodeos span missing entanglement ok log entry:  0
qnodeos span missing entanglement ok log entry:  0
qnodeos span missing entanglement ok log entry:  0
392
400
qnodeos span missing entanglement ok log entry:  0
qnodeos span missing entanglement ok log entry:  0
qnodeos spa

In [142]:
print(len(server_results[0]['results']))

200
